# 01 — Data Load & Clean (NBA-only)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../data/raw/game.csv')
CLEAN_PATH = Path('../data/clean/clean_game_nba.csv')

# --- Config ---
INCLUDE_PLAYOFFS = False  # set True to include playoffs

NBA_TEAMS = [
    "Atlanta Hawks","Boston Celtics","Brooklyn Nets","Charlotte Hornets","Chicago Bulls",
    "Cleveland Cavaliers","Dallas Mavericks","Denver Nuggets","Detroit Pistons","Golden State Warriors",
    "Houston Rockets","Indiana Pacers","LA Clippers","Los Angeles Clippers","Los Angeles Lakers",
    "Memphis Grizzlies","Miami Heat","Milwaukee Bucks","Minnesota Timberwolves","New Orleans Pelicans",
    "New York Knicks","Oklahoma City Thunder","Orlando Magic","Philadelphia 76ers","Phoenix Suns",
    "Portland Trail Blazers","Sacramento Kings","San Antonio Spurs","Toronto Raptors","Utah Jazz"
]

# Load
games = pd.read_csv(RAW_PATH)
# Coerce date
games['game_date'] = pd.to_datetime(games['game_date'], errors='coerce')
# Compute target
games = games.dropna(subset=['pts_home','pts_away'])
games['home_win'] = (games['pts_home'] > games['pts_away']).astype(int)

# Filter to regular season by default
if INCLUDE_PLAYOFFS:
    allowed_types = ['Regular Season','Playoffs']
else:
    allowed_types = ['Regular Season']

mask_type = games['season_type'].isin(allowed_types)
mask_home = games['team_name_home'].isin(NBA_TEAMS)
mask_away = games['team_name_away'].isin(NBA_TEAMS)

nb_games_before = len(games)

games = games[mask_type & mask_home & mask_away].copy()

nb_games_after = len(games)
unique_teams_home = sorted(games['team_name_home'].unique())
unique_teams_away = sorted(games['team_name_away'].unique())

print(f"Rows before filter: {nb_games_before:,}")
print(f"Rows after  filter: {nb_games_after:,}")
print(f"Unique home teams: {len(unique_teams_home)}\nUnique away teams: {len(unique_teams_away)}")
print(sorted(set(unique_teams_home) | set(unique_teams_away)))

# Save cleaned
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
games.to_csv(CLEAN_PATH, index=False)
print(f"Saved cleaned NBA-only data → {CLEAN_PATH}")
